# SILAGEGUARD AI — MobileNetV3-Small Transfer Learning
### SIH26111 — Smart AI-Enabled Rapid Feed and Silage Quality Testing System

This notebook covers:
- MobileNetV3-Small architecture for edge mobile devices
- Albumentations agricultural field data augmentation
- Two-phase transfer learning (head training + full fine-tuning)
- Weighted loss for high recall on 'Unsafe' mold/mycotoxin samples

In [1]:
import os
import sys
import torch
import torchvision
from torchvision import models
import albumentations as A
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Compute Device:', device)

## 1. Load Dataset & Albumentations Augmentations

In [2]:
sys.path.append(os.path.join('..', 'vision_model'))
from dataset_loader import load_dataset_splits, CLASS_NAMES

data_dir = os.path.join('..', 'datasets', 'vision')
train_loader, val_loader, class_names = load_dataset_splits(data_dir, train_ratio=0.8, image_size=224, batch_size=16)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

## 2. Load Pretrained MobileNetV3-Small & Train Head

In [3]:
from train_mobilenetv3 import build_model
model = build_model(num_classes=len(class_names)).to(device)
print(model.classifier)

## 3. Evaluation on Validation Set & Unsafe Recall Check

In [4]:
checkpoint_path = os.path.join('..', 'vision_model', 'mobilenetv3_silage.pth')
if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
    print('Loaded trained checkpoint.')
model.eval()
print('Model ready for edge inference export.')